# 05 - PySpark check of notebook 04 (data quality, RQ1 support tables, RQ2)
**HMDA 2024 project - Group 7 (Mai Anh).** PySpark version of the data-quality checks, the RQ1 support tables and all RQ2 tables of notebook `04_rq2_pandas`, plus the Pandas vs PySpark correctness comparison. Run it **after** notebook 04.

What this notebook does, in order:
1. **Raw data** (`hmda_parquet`, built from `2024_public_lar_one_year_csv.zip` if needed): column/record checks, outcome counts, NA/Exempt profile.
2. **Scope filter** with the record count after every filter, checked against the cleaning log.
3. **Raw-value profile**: missingness by outcome, DTI raw values, quantiles and the outlier counts behind cleaning rules R4-R11.
4. **Cleaned data** (`hmda_clean_v1_spark`, from notebook `01_clean_pyspark`).
5. **The same 37 tables as notebook 04 in Spark**: data quality and missing values (8), RQ1 support (10), RQ2 by loan type (10) and RQ2 by geography (9).
6. **Correctness: Pandas vs PySpark** for every table (counts exact, other numbers within a relative tolerance of 1e-9), saved as `pandas_spark_correctness_rq2.csv`.

**Run every cell from top to bottom in one session (Runtime -> Restart session and run all).** Each check prints `OK` or `MISMATCH`.

> Fixes from the previous version: the old notebook read `2024_combined_mlar.zip` (Modified LAR: no header row, so the first data row became the column names and every column looked "missing"). This version reads the group's public LAR file. The `NameError: scope / sc` errors came from running cells out of order after a restart; every cell here only uses names defined in the cells above it.

## Step 1. Mount Drive and set the paths

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import os, glob, json, math, shutil, time, zipfile

def find_hmda_dir():
    # The group folder "hmda" (shared by Ly) is the folder that contains hmda_clean_v1_spark.
    # Look for it in My Drive (a shortcut) up to 3 levels deep, and in shared drives.
    if os.environ.get("HMDA_DRIVE_DIR"):
        return os.environ["HMDA_DRIVE_DIR"]
    for pattern in ["/content/drive/MyDrive/hmda", "/content/drive/MyDrive/*", "/content/drive/MyDrive/*/*",
                    "/content/drive/MyDrive/*/*/*", "/content/drive/Shareddrives/*", "/content/drive/Shareddrives/*/*"]:
        for p in sorted(glob.glob(pattern)):
            if os.path.isdir(f"{p}/hmda_clean_v1_spark"):
                return p
    return None

DRIVE_DIR = find_hmda_dir()
assert DRIVE_DIR, ("Group folder 'hmda' not found in Drive. Open drive.google.com -> 'Shared with me' -> "
                   "right-click the folder 'hmda' -> Organize -> Add shortcut -> My Drive, then run this cell again.")
LOCAL_DIR     = os.environ.get("HMDA_LOCAL_DIR", "/content/work")      # fast local disk on Colab

RAW_ZIP       = f"{DRIVE_DIR}/2024_public_lar_one_year_csv.zip"    # correct source (NOT 2024_combined_mlar.zip)
PARQUET_DRIVE = f"{DRIVE_DIR}/hmda_parquet"                        # raw data already converted to Parquet
CLEAN_DRIVE   = f"{DRIVE_DIR}/hmda_clean_v1_spark"                 # output of 01_clean_pyspark
RESULTS_DIR   = f"{DRIVE_DIR}/results"
OUT_DRIVE     = f"{RESULTS_DIR}/rq2_spark"                         # outputs of this notebook

PARQUET_LOCAL = f"{LOCAL_DIR}/hmda_parquet"
CLEAN_LOCAL   = f"{LOCAL_DIR}/hmda_clean_v1_spark"
PANDAS_LOCAL  = f"{LOCAL_DIR}/pandas_outputs"
OUT_LOCAL     = f"{LOCAL_DIR}/rq2_spark"
for d in (LOCAL_DIR, OUT_LOCAL):
    os.makedirs(d, exist_ok=True)

print("Group folder found:", DRIVE_DIR)
print("Contents:", sorted(os.listdir(DRIVE_DIR)))

Group folder found: /content/drive/MyDrive/hmda
Contents: ['2024_public_lar_one_year_csv.zip', 'hmda_clean_v1', 'hmda_clean_v1_spark', 'hmda_parquet', 'results', 'tổng hợp code']


## Step 2. Start Spark

In [3]:
!pip install -q pyspark==4.0.5   # same version as requirements.txt

In [4]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder.master("local[*]")
         .config("spark.driver.memory", "8g")
         .appName("hmda_profiling_rq2_pyspark").getOrCreate())
print("Spark", spark.version, "| pandas", pd.__version__)

CHECKS = []          # every check of this notebook, printed at the end

def check(name, got, expected):
    ok = got == expected
    CHECKS.append({"check": name, "got": got, "expected": expected, "status": "OK" if ok else "MISMATCH"})
    print(f"{'OK      ' if ok else 'MISMATCH'}  {name:44s} got={got:>12,}  expected={expected:>12,}")

Spark 4.0.5 | pandas 2.2.3


## Step 3. Raw data: load and check
Uses `hmda_parquet` from Drive. If it is not there, it is built once from `2024_public_lar_one_year_csv.zip`: the zip is copied to the local disk, the delimiter is read from the header line, and the file is converted to Parquet (all columns kept as strings, like the group's version).

In [5]:
def build_parquet_from_zip(zip_path, out_dir):
    work = f"{LOCAL_DIR}/hmda_raw"
    os.makedirs(work, exist_ok=True)
    local_zip = f"{work}/{os.path.basename(zip_path)}"
    if not os.path.exists(local_zip):
        shutil.copy(zip_path, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(work)
    files = [f for f in glob.glob(f"{work}/**/*", recursive=True) if f.lower().endswith((".csv", ".txt"))]
    assert files, "no .csv/.txt file inside the zip"
    path = files[0]
    with open(path, encoding="utf-8", errors="replace") as fh:
        header = fh.readline()
    assert "action_taken" in header, "first line has no column names - wrong file? (the MLAR file has no header row)"
    sep = "|" if header.count("|") > header.count(",") else ","
    print(f"Converting {os.path.basename(path)} (delimiter '{sep}') to Parquet ...")
    (spark.read.option("header", True).option("sep", sep).option("inferSchema", False).csv(path)
          .write.mode("overwrite").parquet(out_dir))

if not os.path.exists(PARQUET_LOCAL):
    if os.path.exists(PARQUET_DRIVE):
        shutil.copytree(PARQUET_DRIVE, PARQUET_LOCAL)
    else:
        assert os.path.exists(RAW_ZIP), f"neither {PARQUET_DRIVE} nor {RAW_ZIP} found"
        build_parquet_from_zip(RAW_ZIP, PARQUET_LOCAL)

raw = spark.read.parquet(PARQUET_LOCAL)
n_total = raw.count()
print("First columns:", raw.columns[:8])
check("raw: number of columns", len(raw.columns), 99)
check("raw: number of records", n_total, 12259199)

NEEDED = ["action_taken", "occupancy_type", "loan_purpose", "lien_status", "derived_dwelling_category",
          "reverse_mortgage", "open_end_line_of_credit", "business_or_commercial_purpose",
          "derived_msa_md", "state_code", "loan_type", "income", "debt_to_income_ratio", "loan_amount",
          "combined_loan_to_value_ratio", "property_value", "loan_term"]
missing_cols = [c for c in NEEDED if c not in raw.columns]
print("Missing columns:", missing_cols)
assert not missing_cols, "columns missing - wrong file or wrong delimiter, stop here"

First columns: ['activity_year', 'lei', 'derived_msa_md', 'state_code', 'county_code', 'census_tract', 'conforming_loan_limit', 'derived_loan_product_type']
OK        raw: number of columns                       got=          99  expected=          99
OK        raw: number of records                       got=  12,259,199  expected=  12,259,199
Missing columns: []


In [6]:
# Outcome codes in the raw file (1-3 are used, 4-8 are tabulated only)
raw.groupBy("action_taken").count().orderBy("action_taken").show()

+------------+-------+
|action_taken|  count|
+------------+-------+
|           1|6197085|
|           2| 361232|
|           3|2103462|
|           4|1538273|
|           5| 581940|
|           6|1276504|
|           7|  47312|
|           8| 153391|
+------------+-------+



In [7]:
# Special values in the raw file: null, and the HMDA codes NA / Exempt / 1111
PROFILE_COLS = ["income", "debt_to_income_ratio", "loan_amount", "combined_loan_to_value_ratio",
                "property_value", "loan_term", "interest_rate", "rate_spread", "derived_msa_md", "state_code"]
PROFILE_COLS = [c for c in PROFILE_COLS if c in raw.columns]
exprs = []
for c in PROFILE_COLS:
    exprs += [F.count(F.when(F.col(c).isNull(), 1)).alias(c + "__null"),
              F.count(F.when(F.col(c).isin("NA", "Exempt", "1111"), 1)).alias(c + "__special")]
row = raw.agg(*exprs).collect()[0].asDict()
pd.DataFrame([{"column": c, "null": row[c + "__null"], "NA_Exempt_1111": row[c + "__special"]} for c in PROFILE_COLS])

,column,null,NA_Exempt_1111
0,income,0,1780511
1,debt_to_income_ratio,0,4347476
2,loan_amount,0,0
3,combined_loan_to_value_ratio,0,4318147
4,property_value,0,2744075
5,loan_term,0,491153
6,interest_rate,0,4519895
7,rate_spread,0,6429918
8,derived_msa_md,0,0
9,state_code,0,215946


## Step 4. Scope filter
Owner-occupied, first-lien, site-built 1-4 unit home-purchase applications with outcome 1-3, excluding reverse-mortgage, open-end and business-purpose records. Every count is checked against `cleaning_log_spark.json`.

In [8]:
SCOPE_VALUES = {
    "action_taken": ["1", "2", "3"],
    "occupancy_type": ["1"],
    "loan_purpose": ["1"],
    "lien_status": ["1"],
    "derived_dwelling_category": ["Single Family (1-4 Units):Site-Built"],
    "reverse_mortgage": ["2"],
    "open_end_line_of_credit": ["2"],
    "business_or_commercial_purpose": ["2"],
}
EXPECTED_STEPS = {"action_taken": 8661779, "occupancy_type": 7792568, "loan_purpose": 3811298,
                  "lien_status": 3641962, "derived_dwelling_category": 3237835, "reverse_mortgage": 3168105,
                  "open_end_line_of_credit": 3145390, "business_or_commercial_purpose": 3140857}

scope = raw
for col, vals in SCOPE_VALUES.items():
    scope = scope.filter(F.col(col).isin(vals))
    check(f"scope: after filter {col}", scope.count(), EXPECTED_STEPS[col])

scope = scope.select(*NEEDED).cache()     # keep only the columns used below (lighter than all 99)
outcome = {r["action_taken"]: r["count"] for r in scope.groupBy("action_taken").count().collect()}
check("scope: originated (1)", outcome.get("1", 0), 2775269)
check("scope: approved, not accepted (2)", outcome.get("2", 0), 88984)
check("scope: denied (3)", outcome.get("3", 0), 276604)

OK        scope: after filter action_taken             got=   8,661,779  expected=   8,661,779
OK        scope: after filter occupancy_type           got=   7,792,568  expected=   7,792,568
OK        scope: after filter loan_purpose             got=   3,811,298  expected=   3,811,298
OK        scope: after filter lien_status              got=   3,641,962  expected=   3,641,962
OK        scope: after filter derived_dwelling_category got=   3,237,835  expected=   3,237,835
OK        scope: after filter reverse_mortgage         got=   3,168,105  expected=   3,168,105
OK        scope: after filter open_end_line_of_credit  got=   3,145,390  expected=   3,145,390
OK        scope: after filter business_or_commercial_purpose got=   3,140,857  expected=   3,140,857
OK        scope: originated (1)                        got=   2,775,269  expected=   2,775,269
OK        scope: approved, not accepted (2)            got=      88,984  expected=      88,984
OK        scope: denied (3)                

In [9]:
# Share of NA/Exempt by outcome (the proposal reports CLTV: 2.7% originated, 7.0% approved-not-accepted, 5.5% denied)
for c in ["debt_to_income_ratio", "income", "combined_loan_to_value_ratio", "property_value"]:
    print("==", c)
    (scope.groupBy("action_taken")
          .agg(F.count("*").alias("n"),
               F.round(100 * F.count(F.when(F.col(c).isin("NA", "Exempt"), 1)) / F.count("*"), 2).alias("pct_NA_Exempt"))
          .orderBy("action_taken").show())

print("== 5 largest MSA/MD codes (99999 = outside any MSA/MD)")
scope.groupBy("derived_msa_md").count().orderBy(F.desc("count")).show(5)
print("== DTI raw values (mixed format: ranges as text and single values 36-49)")
scope.groupBy("debt_to_income_ratio").count().orderBy(F.desc("count")).show(25, truncate=False)

== debt_to_income_ratio
+------------+-------+-------------+
|action_taken|      n|pct_NA_Exempt|
+------------+-------+-------------+
|           1|2775269|         0.43|
|           2|  88984|         5.25|
|           3| 276604|         3.12|
+------------+-------+-------------+

== income
+------------+-------+-------------+
|action_taken|      n|pct_NA_Exempt|
+------------+-------+-------------+
|           1|2775269|         0.67|
|           2|  88984|         0.58|
|           3| 276604|         0.79|
+------------+-------+-------------+

== combined_loan_to_value_ratio
+------------+-------+-------------+
|action_taken|      n|pct_NA_Exempt|
+------------+-------+-------------+
|           1|2775269|         2.72|
|           2|  88984|         6.96|
|           3| 276604|         5.52|
+------------+-------+-------------+

== property_value
+------------+-------+-------------+
|action_taken|      n|pct_NA_Exempt|
+------------+-------+-------------+
|           1|2775269|   

## Step 5. Numeric values before cleaning: quantiles and outlier counts
Values are cast with `try_cast` (Spark 4 runs in ANSI mode), so NA / Exempt become null, never 0. The counts are the ones behind cleaning rules R4-R11 in `01_clean_pyspark`.

In [10]:
NUM_COLS = ["loan_amount", "income", "combined_loan_to_value_ratio", "property_value", "loan_term"]
sc = scope.select("action_taken", *[F.expr(f"try_cast(`{c}` as double)").alias(c) for c in NUM_COLS])
sc = sc.withColumn("loan_to_income",
                   F.when(F.col("income") > 0, F.expr("try_divide(loan_amount, income * 1000)")))   # income <= 0 is invalid
sc = sc.cache()

QCOLS = NUM_COLS + ["loan_to_income"]
qs = sc.approxQuantile(QCOLS, [0.0, 0.01, 0.25, 0.5, 0.75, 0.99, 1.0], 0.001)
pd.DataFrame(qs, index=QCOLS, columns=["min", "p1", "p25", "median", "p75", "p99", "max"])   # approximate, for a first look

,min,p1,p25,median,p75,p99,max
loan_amount,5000.000000,75000.000000,225000.000000,325000.000000,475000.000000,1.535000e+06,3.010500e+07
income,-19288.000000,24.000000,75.000000,111.000000,170.000000,7.250000e+02,7.212000e+06
combined_loan_to_value_ratio,0.031000,26.104000,80.000000,90.000000,96.500000,1.030020e+02,2.250000e+09
property_value,5000.000000,95000.000000,265000.000000,385000.000000,575000.000000,2.205000e+06,2.147484e+09
loan_term,1.000000,180.000000,360.000000,360.000000,360.000000,3.600000e+02,3.292000e+03
loan_to_income,0.000064,0.656566,2.204611,2.946429,3.760331,6.376147e+00,2.255000e+03


In [11]:
INT32_MAX = 2147483647
o = sc.agg(
    *[F.count(F.when(F.col(c).isNull(), 1)).alias(c + "_null") for c in QCOLS],
    F.count(F.when(F.col("income") <= 0, 1)).alias("income_le0"),
    F.count(F.when(F.col("loan_amount") <= 0, 1)).alias("loan_le0"),
    F.count(F.when(F.col("combined_loan_to_value_ratio") > 100, 1)).alias("cltv_gt100"),
    F.count(F.when(F.col("combined_loan_to_value_ratio") > 200, 1)).alias("cltv_gt200"),
    F.count(F.when(F.col("combined_loan_to_value_ratio") < 5, 1)).alias("cltv_lt5"),
    F.count(F.when(F.col("property_value") == INT32_MAX, 1)).alias("pv_int_max"),
    F.count(F.when(F.col("loan_term") < 12, 1)).alias("term_lt12"),
    F.count(F.when(F.col("loan_term") > 480, 1)).alias("term_gt480"),
    F.count(F.when(F.col("income") > 5000, 1)).alias("income_gt5000"),
    F.count(F.when(F.col("loan_to_income") > 10, 1)).alias("lti_gt10"),
).collect()[0].asDict()

for k in ["loan_le0", "cltv_gt100"]:                      # shown only, not a cleaning rule
    print(f"info      {k:44s} {o[k]:>12,}")
check("R4  income <= 0 (invalid)", o["income_le0"], 16942)
check("R5  CLTV > 200 (invalid)", o["cltv_gt200"], 371)
check("R5  CLTV < 5 (invalid)", o["cltv_lt5"], 288)
check("R6  property_value = 2^31-1 (invalid)", o["pv_int_max"], 2)
check("R7  loan_term < 12 (invalid)", o["term_lt12"], 1001)
check("R7  loan_term > 480 (invalid)", o["term_gt480"], 4)
check("R11 income > 5000 (kept, flagged)", o["income_gt5000"], 857)
check("R10 loan-to-income > 10 (kept, flagged)", o["lti_gt10"], 11937)
_ = (sc.unpersist(), scope.unpersist())   # free memory before loading the cleaned data

info      loan_le0                                                0
info      cltv_gt100                                        142,573
OK        R4  income <= 0 (invalid)                    got=      16,942  expected=      16,942
OK        R5  CLTV > 200 (invalid)                     got=         371  expected=         371
OK        R5  CLTV < 5 (invalid)                       got=         288  expected=         288
OK        R6  property_value = 2^31-1 (invalid)        got=           2  expected=           2
OK        R7  loan_term < 12 (invalid)                 got=       1,001  expected=       1,001
OK        R7  loan_term > 480 (invalid)                got=           4  expected=           4
OK        R11 income > 5000 (kept, flagged)            got=         857  expected=         857
OK        R10 loan-to-income > 10 (kept, flagged)      got=      11,937  expected=      11,937


## Step 6. Cleaned data (`hmda_clean_v1_spark`)
Output of `01_clean_pyspark` (rules v1, already checked equal to the Pandas version). The folder also holds `cleaning_log_spark.json`, so only the `.parquet` files are read.

Note: the copy of `cleaning_log_spark.json` inside this folder has no duplicate count. That is expected: the optional duplicate cell (rule R12, 190 duplicates over all 99 columns) writes only to `results/cleaning_log_spark.json`.

In [12]:
if not os.path.exists(CLEAN_LOCAL):
    shutil.copytree(CLEAN_DRIVE, CLEAN_LOCAL)
clean = spark.read.parquet(f"{CLEAN_LOCAL}/*.parquet")
print(clean.columns)

totals = clean.agg(F.count("*").alias("n"), F.sum("denied").alias("denied")).collect()[0]
check("clean: rows", totals["n"], 3140857)
check("clean: denied rows", int(totals["denied"]), 276604)
for c in ["income_band", "loan_amount_band", "cltv_band", "dti_band", "census_region", "loan_to_income"]:
    print(f"{'present ' if c in clean.columns else 'MISSING '} {c}")

['action_taken', 'denied', 'state_code', 'census_region', 'derived_msa_md', 'msa_md_clean', 'metro_flag', 'loan_type', 'lei', 'loan_amount', 'income', 'combined_loan_to_value_ratio', 'property_value', 'loan_term', 'loan_to_income', 'dti_band', 'income_band', 'loan_amount_band', 'cltv_band', 'income_invalid', 'cltv_invalid', 'property_value_invalid', 'loan_term_invalid', 'income_gt5000', 'lti_gt10', 'val_race', 'val_ethnicity', 'val_sex', 'val_age']
OK        clean: rows                                  got=   3,140,857  expected=   3,140,857
OK        clean: denied rows                           got=     276,604  expected=     276,604
present  income_band
present  loan_amount_band
present  cltv_band
present  dti_band
present  census_region
present  loan_to_income


## Step 7. Definitions (same as notebook 04)
- Denial rate in %, **Wilson 95% interval** with z = 1.959963984540054 (`scipy.stats.norm.ppf(0.975)`); `reportable` = at least 100 records.
- **Reference = pooled rate of exactly the group being compared**: between states **50 states + DC (8.62%)**; between regions, loan types, metro status and MSAs **all records (8.81%)**. Raw gap = rate of the group minus its reference (pp).
- Medians and quartiles are **exact**, over observed values only. Spark's `percentile()` uses the same linear interpolation as Pandas `quantile()`.
- Bands: those of the cleaning rules (`income_band`, `loan_amount_band`, `cltv_band`, `dti_band`), not re-binned. DTI shares (<36, 36-<50, 50+) in the profiles: denominator = records **with a known DTI** (0.80% missing left out); `share_dti_missing_pct` and `rq2_region_dti_distribution` use all records. Correlations across states: Spearman rho is the main number.
- Geography: **50 states + DC**; territories and records without a state reported separately (rules of notebook 02_rq1_pandas). MSAs: codes other than 99999 with **at least 5,000 records**.

In [13]:
Z = 1.959963984540054
MIN_N = 100
MIN_CELL = 500
MSA_MIN_N = 5000      # same cut-off as notebook 02_rq1_pandas (MSAs with n >= 5,000)
GAP_MIN_N = 10000
FOUR_REGIONS = ["Northeast", "Midwest", "South", "West"]
ALL_STATES = "All 50 states + DC"          # reference row of state comparisons (8.62%)
ALL_RECORDS = "All records"                # reference row of region comparisons (8.81%)
REGION_ROWS = FOUR_REGIONS + ["Territory"]
LOAN_TYPE_NAMES = {"1": "Conventional", "2": "FHA", "3": "VA", "4": "USDA RHS/FSA"}
DTI_FOCUS = ["<20", "45-<50", "50-60", ">60", "Missing"]
DTI_GROUPS = {"lt36": ["<20", "20-<30", "30-<36"], "36_50": ["36-<40", "40-<45", "45-<50"], "50plus": ["50-60", ">60"]}
LT_SHARE = {"conventional": "Conventional", "fha": "FHA", "va": "VA", "usda": "USDA RHS/FSA"}
METRICS = {"income_kusd": "income", "loan_usd": "loan_amount", "cltv_pct": "combined_loan_to_value_ratio"}
INDICATORS = ["income_kusd_median", "loan_usd_median", "lti_median",
              "share_dti_lt36_pct", "share_dti_36_50_pct", "share_dti_50plus_pct", "share_dti_missing_pct",
              "share_conventional_pct", "share_fha_pct", "share_va_pct", "share_usda_pct"]

def mapped(col, mapping):
    return F.create_map([F.lit(x) for kv in mapping.items() for x in kv])[F.col(col)]

d = (clean
     .withColumn("loan_type_name", mapped("loan_type", LOAN_TYPE_NAMES))
     .withColumn("decision_group", F.when(F.col("denied") == 1, "Denied").otherwise("Originated/approved"))
     .withColumn("dti_cat", F.col("dti_band"))
     .withColumn("geo_group", F.when(F.col("census_region").isin(FOUR_REGIONS), ALL_STATES)
                               .when(F.col("census_region") == "Territory", "Territory").otherwise("Missing state"))
     .withColumn("metro_status", F.when(F.col("metro_flag") == 1, "Metro").otherwise("Non-metro"))
     .withColumnRenamed("state_code", "state").withColumnRenamed("derived_msa_md", "msa")
    ).cache()
d50 = d.filter(F.col("geo_group") == ALL_STATES)
dreg = d.filter(F.col("geo_group") != "Missing state")       # records with a region (4 regions + Territory)
tot = d.agg(F.count(F.lit(1)).alias("n"), F.sum("denied").alias("k")).collect()[0]
t50 = d50.agg(F.count(F.lit(1)).alias("n"), F.sum("denied").alias("k")).collect()[0]
NAT_PCT = 100 * tot["k"] / tot["n"]                     # all records (8.81%): reference between regions, loan types, metro, MSAs
STATE_REF_PCT = 100 * t50["k"] / t50["n"]               # 50 states + DC (8.62%): reference between states
check("denied in 50 states + DC", int(t50["k"]), 268873)
print(f"Rows: {tot['n']:,} | loan types without a name: {d.filter(F.col('loan_type_name').isNull()).count()} | "
      f"all records {NAT_PCT:.4f}% | 50 states + DC {STATE_REF_PCT:.4f}%")

def rate_ci(g):
    # denial rate (%) with Wilson 95% interval and the reportable flag
    n, k = F.col("n").cast("double"), F.col("denied").cast("double")
    p = k / n
    den = 1 + Z * Z / n
    centre = (p + Z * Z / (2 * n)) / den
    half = Z * F.sqrt(p * (1 - p) / n + Z * Z / (4 * n * n)) / den
    return (g.withColumn("denial_rate_pct", 100 * p)
             .withColumn("wilson_95_low_pct", 100 * (centre - half))
             .withColumn("wilson_95_high_pct", 100 * (centre + half))
             .withColumn("reportable", F.col("n") >= MIN_N))

def rate_table(df, keys):
    g = df.groupBy(*keys).agg(F.count(F.lit(1)).alias("n"), F.sum("denied").cast("long").alias("denied"))
    return rate_ci(g).orderBy(*keys)

def financial_profile(df, keys, with_rate=False):
    aggs = [F.count(F.lit(1)).alias("n"), F.sum("denied").cast("long").alias("denied")]
    for name, col in METRICS.items():
        aggs += [F.count(col).alias(f"{name}_observed_n"),
                 F.expr(f"percentile(`{col}`, array(0.25, 0.5, 0.75))").alias(f"_{name}_q")]
    g = df.groupBy(*keys).agg(*aggs)
    cols = list(keys) + ["n", "denied"]
    for name in METRICS:
        g = (g.withColumn(f"{name}_p25", F.col(f"_{name}_q")[0])
              .withColumn(f"{name}_median", F.col(f"_{name}_q")[1])
              .withColumn(f"{name}_p75", F.col(f"_{name}_q")[2])
              .withColumn(f"{name}_missing_pct", 100 * (F.col("n") - F.col(f"{name}_observed_n")) / F.col("n")))
        cols += [f"{name}_observed_n", f"{name}_p25", f"{name}_median", f"{name}_p75", f"{name}_missing_pct"]
    g = g.select(*cols)
    if with_rate:
        g = rate_ci(g)
    return g.orderBy(*keys)

def indicator_profile(df, keys, ref):
    known = F.col("dti_cat") != "Missing"
    share = lambda cond: 100 * F.avg(F.when(cond, 1.0).otherwise(0.0))
    share_known = lambda cond: 100 * F.avg(F.when(known, F.when(cond, 1.0).otherwise(0.0)))   # null when DTI missing
    g = df.groupBy(*keys).agg(
        F.count(F.lit(1)).alias("n"), F.sum("denied").cast("long").alias("denied"),
        F.expr("percentile(income, 0.5)").alias("income_kusd_median"),
        F.expr("percentile(loan_amount, 0.5)").alias("loan_usd_median"),
        F.expr("percentile(loan_to_income, 0.5)").alias("lti_median"),
        *[share_known(F.col("dti_cat").isin(v)).alias(f"share_dti_{k}_pct") for k, v in DTI_GROUPS.items()],
        share(~known).alias("share_dti_missing_pct"),
        *[share(F.col("loan_type_name") == v).alias(f"share_{k}_pct") for k, v in LT_SHARE.items()])
    g = (g.withColumn("denial_rate_pct", 100 * F.col("denied") / F.col("n"))
          .withColumn("gap_pp", F.col("denial_rate_pct") - F.lit(ref)))
    return g.select(*keys, "n", "denied", "denial_rate_pct", "gap_pp", *INDICATORS)

def share_table(df, keys, cat):
    w = Window.partitionBy(*keys)
    t = rate_table(df, keys + [cat]).withColumn("share_pct", 100 * F.col("n") / F.sum("n").over(w))
    return t.select(*keys, cat, "n", "share_pct", "denied", "denial_rate_pct", "wilson_95_low_pct", "wilson_95_high_pct", "reportable")

# The three functions below work on small Spark result tables after toPandas(): same code as notebook 04
def corr_with_gap(states):
    rows = []
    y, w = states["gap_pp"].to_numpy(float), states["n"].to_numpy(float)
    for c in INDICATORS:
        x = states[c].to_numpy(float)
        ok = ~np.isnan(x)
        xs, ys, ws = x[ok], y[ok], w[ok]
        pear = float(np.corrcoef(xs, ys)[0, 1])
        spear = float(pd.Series(xs).rank().corr(pd.Series(ys).rank()))
        mx, my = np.average(xs, weights=ws), np.average(ys, weights=ws)
        cov = np.average((xs - mx) * (ys - my), weights=ws)
        wpear = float(cov / np.sqrt(np.average((xs - mx) ** 2, weights=ws) * np.average((ys - my) ** 2, weights=ws)))
        rows.append({"indicator": c, "n_states": int(ok.sum()), "spearman_rho": spear, "pearson_r": pear,
                     "weighted_pearson_r": wpear})
    return pd.DataFrame(rows)

def extreme_states(states):
    eligible = states[states["n"] >= GAP_MIN_N]
    top = eligible.sort_values(["gap_pp", "state"], ascending=[False, True]).head(5)["state"].tolist()
    rest = eligible[~eligible["state"].isin(top)]
    bottom = rest.sort_values(["gap_pp", "state"], ascending=[True, True]).head(5)["state"].tolist()
    return top, bottom

def mix_summary(prev):
    rows = []
    for mix, col in [("loan type", "exp_gap_loan_type_pp"), ("loan type x DTI band", "exp_gap_loan_type_dti_pp")]:
        res = col.replace("exp_", "resid_")
        rows.append({"mix": mix, "n_states": len(prev),
                     "r_squared": float(np.corrcoef(prev["raw_gap_pp"], prev[col])[0, 1] ** 2),
                     "spearman_rho": float(prev["raw_gap_pp"].rank().corr(prev[col].rank())),
                     "mean_abs_raw_gap_pp": float(prev["raw_gap_pp"].abs().mean()),
                     "mean_abs_remaining_gap_pp": float(prev[res].abs().mean())})
    return pd.DataFrame(rows)
print("Helpers defined")

OK        denied in 50 states + DC                     got=     268,873  expected=     268,873
Rows: 3,140,857 | loan types without a name: 0 | all records 8.8066% | 50 states + DC 8.6179%
Helpers defined


## Step 8. Data-quality tables in Spark (including missing values and denial)

In [14]:
SPARK_TABLES = {}       # table name -> (Pandas DataFrame from Spark, key columns)
t_start = time.perf_counter()

SPARK_TABLES["cohort_outcome_counts"] = (
    d.groupBy("action_taken").agg(F.count(F.lit(1)).alias("n"), F.sum("denied").cast("long").alias("denied"))
     .orderBy("action_taken").toPandas(), ["action_taken"])

m = d.agg(F.count(F.lit(1)).alias("rows"),
          *[F.sum(F.col(c).isNull().cast("int")).alias(name) for name, c in METRICS.items()],
          F.sum((F.col("dti_cat") == "Missing").cast("int")).alias("dti")).collect()[0]
rows = [{"field": name, "missing_n": int(m[name])} for name in METRICS] + [{"field": "dti_cat (Missing)", "missing_n": int(m["dti"])}]
miss = pd.DataFrame(rows)
miss["missing_pct"] = 100 * miss["missing_n"] / m["rows"]
miss["rows"] = int(m["rows"])
SPARK_TABLES["data_quality_missingness"] = (miss, ["field"])

dti = (d.groupBy("dti_cat").agg(F.count(F.lit(1)).alias("n"), F.sum("denied").cast("long").alias("denied"))
        .withColumn("denial_rate_pct", 100 * F.col("denied") / F.col("n")).toPandas())
SPARK_TABLES["data_quality_dti_overall"] = (dti, ["dti_cat"])

qrows = []
for name, c in METRICS.items():
    r = d.agg(F.count(c).alias("n_observed"), F.min(c).alias("min"),
              F.expr(f"percentile(`{c}`, array(0.01, 0.25, 0.5, 0.75, 0.99))").alias("q"),
              F.max(c).alias("max")).collect()[0]
    qrows.append({"field": name, "n_observed": r["n_observed"], "min": r["min"], "p01": r["q"][0], "p25": r["q"][1],
                  "median": r["q"][2], "p75": r["q"][3], "p99": r["q"][4], "max": r["max"]})
SPARK_TABLES["data_quality_numeric_quantiles"] = (pd.DataFrame(qrows), ["field"])

FLAGS = ["income_invalid", "cltv_invalid", "property_value_invalid", "loan_term_invalid", "income_gt5000", "lti_gt10"]
f = d.agg(*[F.sum(c).cast("long").alias(c) for c in FLAGS]).collect()[0]
SPARK_TABLES["data_quality_original_flags"] = (pd.DataFrame({"flag": FLAGS, "flagged_n": [int(f[c]) for c in FLAGS]}), ["flag"])

# Missing values and denial
MISS_FIELDS = {"Income": F.col("income").isNull(), "CLTV": F.col("combined_loan_to_value_ratio").isNull(),
               "Loan-to-income": F.col("loan_to_income").isNull(), "DTI": F.col("dti_cat") == "Missing"}
rows = []
for name, cond in MISS_FIELDS.items():
    r = d.agg(F.sum(cond.cast("int")).alias("mn"), F.avg(cond.cast("double")).alias("mp"),
              F.avg(F.when(cond, F.col("denied").cast("double"))).alias("rm"),
              F.avg(F.when(~cond, F.col("denied").cast("double"))).alias("ro")).collect()[0]
    rows.append({"field": name, "missing_n": int(r["mn"]), "missing_pct": 100 * r["mp"],
                 "denial_rate_if_missing_pct": 100 * r["rm"], "denial_rate_if_observed_pct": 100 * r["ro"]})
mv = pd.DataFrame(rows)
mv["ratio"] = mv["denial_rate_if_missing_pct"] / mv["denial_rate_if_observed_pct"]
SPARK_TABLES["rq2_missingness_vs_denial"] = (mv, ["field"])

cm = (d.groupBy("action_taken").agg(F.count(F.lit(1)).alias("n"),
                                    F.sum(F.col("combined_loan_to_value_ratio").isNull().cast("long")).alias("cltv_missing_n"))
       .withColumn("cltv_missing_pct", 100 * F.col("cltv_missing_n") / F.col("n")).orderBy("action_taken").toPandas())
SPARK_TABLES["rq2_cltv_missing_by_outcome"] = (cm, ["action_taken"])

complete = F.col("income").isNotNull() & F.col("combined_loan_to_value_ratio").isNotNull() & (F.col("dti_cat") != "Missing")
ref_cc = 100 * d.filter(complete).agg(F.avg(F.col("denied").cast("double"))).first()[0]
by = (dreg.withColumn("_cc", complete).groupBy("census_region")
         .agg(F.count(F.lit(1)).alias("n_all"), F.avg(F.col("denied").cast("double")).alias("ra"),
              F.sum(F.col("_cc").cast("long")).alias("n_complete"),
              F.avg(F.when(F.col("_cc"), F.col("denied").cast("double"))).alias("rc")).toPandas().set_index("census_region"))
rows = []
for reg_name in REGION_ROWS + [ALL_RECORDS]:
    if reg_name == ALL_RECORDS:
        r = d.withColumn("_cc", complete).agg(F.count(F.lit(1)).alias("n_all"), F.avg(F.col("denied").cast("double")).alias("ra"),
              F.sum(F.col("_cc").cast("long")).alias("n_complete"),
              F.avg(F.when(F.col("_cc"), F.col("denied").cast("double"))).alias("rc")).first().asDict()
    else:
        r = by.loc[reg_name].to_dict()
    rate_all, rate_cc = 100 * r["ra"], 100 * r["rc"]
    rows.append({"census_region": reg_name, "n_all": int(r["n_all"]), "denial_rate_all_pct": rate_all,
                 "gap_all_pp": rate_all - NAT_PCT, "n_complete": int(r["n_complete"]),
                 "dropped_pct": 100 * (1 - r["n_complete"] / r["n_all"]), "denial_rate_complete_pct": rate_cc,
                 "gap_complete_pp": rate_cc - ref_cc, "change_in_gap_pp": (rate_cc - ref_cc) - (rate_all - NAT_PCT)})
SPARK_TABLES["rq2_complete_case_vs_missing_category"] = (pd.DataFrame(rows), ["census_region"])

for name in ["cohort_outcome_counts", "data_quality_missingness", "rq2_missingness_vs_denial",
             "rq2_cltv_missing_by_outcome", "rq2_complete_case_vs_missing_category"]:
    print("==", name); print(SPARK_TABLES[name][0].round(3).to_string(index=False)); print()

== cohort_outcome_counts
 action_taken       n  denied
            1 2775269       0
            2   88984       0
            3  276604  276604

== data_quality_missingness
            field  missing_n  missing_pct    rows
      income_kusd      38308        1.220 3140857
         loan_usd          0        0.000 3140857
         cltv_pct      97627        3.108 3140857
dti_cat (Missing)      25138        0.800 3140857

== rq2_missingness_vs_denial
         field  missing_n  missing_pct  denial_rate_if_missing_pct  denial_rate_if_observed_pct  ratio
        Income      38308        1.220                      26.052                        8.594  3.032
          CLTV      97627        3.108                      16.026                        8.575  1.869
Loan-to-income      38308        1.220                      26.052                        8.594  3.032
           DTI      25138        0.800                      34.295                        8.601  3.987

== rq2_cltv_missing_by_outcome

## Step 9. RQ1 support and RQ2 tables in Spark
RQ2 by loan type (financial profiles, denial rate by DTI / income / CLTV / loan-amount band, DTI x CLTV), RQ1 support tables with the rules of notebook 02_rq1_pandas, and RQ2 by geography (region and state profiles, gaps, correlations, top-5 vs bottom-5 states, gap expected from mix). Correlations, top/bottom states and the mix summary are computed with the same Pandas code as notebook 04, on the Spark result tables.

In [15]:
SPARK_TABLES["rq2_financial_profiles"] = (
    financial_profile(d, ["loan_type_name", "decision_group"]).toPandas(), ["loan_type_name", "decision_group"])

mo = (d.groupBy("decision_group")
       .agg(F.count(F.lit(1)).alias("n"), F.count("income").alias("income_observed_n"),
            F.count("combined_loan_to_value_ratio").alias("cltv_observed_n"),
            F.sum((F.col("dti_cat") == "Missing").cast("int")).alias("dti_missing_n"))
       .withColumn("income_missing_pct", 100 * (F.col("n") - F.col("income_observed_n")) / F.col("n"))
       .withColumn("cltv_missing_pct", 100 * (F.col("n") - F.col("cltv_observed_n")) / F.col("n"))
       .withColumn("dti_missing_pct", 100 * F.col("dti_missing_n") / F.col("n"))
       .orderBy("decision_group"))
SPARK_TABLES["rq2_missingness_by_outcome"] = (mo.toPandas(), ["decision_group"])

for name, band in [("rq2_dti_by_loan_type", "dti_cat"), ("rq2_income_by_loan_type", "income_band"),
                   ("rq2_cltv_by_loan_type", "cltv_band"), ("rq2_loan_amount_by_loan_type", "loan_amount_band")]:
    SPARK_TABLES[name] = (rate_table(d, ["loan_type_name", band]).toPandas(), ["loan_type_name", band])
dti_lt = SPARK_TABLES["rq2_dti_by_loan_type"][0]
SPARK_TABLES["rq2_dti_interpretation_focus"] = (dti_lt[dti_lt["dti_cat"].isin(DTI_FOCUS)].reset_index(drop=True),
                                                ["loan_type_name", "dti_cat"])
SPARK_TABLES["rq2_dti_x_cltv"] = (rate_table(d, ["dti_cat", "cltv_band"]).withColumn("shown", F.col("n") >= MIN_CELL).toPandas(),
                                  ["dti_cat", "cltv_band"])

states = financial_profile(d50, ["state"], with_rate=True)
SPARK_TABLES["rq2_state_financial_profiles"] = (states.orderBy(F.desc("denial_rate_pct")).toPandas(), ["state"])
msa = financial_profile(d.filter(F.col("msa") != "99999"), ["msa"], with_rate=True).filter(F.col("n") >= MSA_MIN_N)
SPARK_TABLES["rq2_msa_financial_profiles"] = (msa.orderBy(F.desc("n")).toPandas(), ["msa"])

# ---- RQ1 support tables (rules of notebook 02_rq1_pandas)
geo = d.filter(F.col("geo_group") != "Missing state")
SPARK_TABLES["rq1_loan_type_rates"] = (rate_table(d, ["loan_type_name"]).toPandas(), ["loan_type_name"])
SPARK_TABLES["rq1_metro_status_rates"] = (rate_table(d, ["metro_status"]).toPandas(), ["metro_status"])
SPARK_TABLES["rq1_region_rates"] = (rate_table(geo, ["census_region"]).toPandas(), ["census_region"])
SPARK_TABLES["rq1_region_x_loan_type"] = (rate_table(geo, ["census_region", "loan_type_name"]).toPandas(),
                                          ["census_region", "loan_type_name"])
st_all = rate_table(geo, ["geo_group", "state"]).toPandas()
SPARK_TABLES["rq1_state_all"] = (st_all, ["geo_group", "state"])
SPARK_TABLES["rq1_state_reportable"] = (st_all[st_all["geo_group"] == ALL_STATES].drop(columns="geo_group"), ["state"])
SPARK_TABLES["rq1_territories"] = (st_all[st_all["geo_group"] == "Territory"].drop(columns="geo_group"), ["state"])
ms = (rate_table(d.filter(F.col("geo_group") == "Missing state").withColumn("group", F.lit("Missing state (state_code = NA)")),
                 ["group"]).withColumn("share_of_cohort_pct", 100 * F.col("n") / F.lit(tot["n"])))
SPARK_TABLES["rq1_state_missing_summary"] = (ms.toPandas(), ["group"])
msa_all = rate_table(d, ["msa"]).toPandas()
SPARK_TABLES["rq1_msa_all"] = (msa_all, ["msa"])
SPARK_TABLES["rq1_msa_reportable"] = (msa_all[(msa_all["msa"] != "99999") & (msa_all["n"] >= MSA_MIN_N)], ["msa"])

# ---- RQ2 by geography and the link to raw gaps
d_all = d.withColumn("census_region", F.lit(ALL_RECORDS))
reg = pd.concat([indicator_profile(dreg, ["census_region"], NAT_PCT).toPandas(),
                 indicator_profile(d_all, ["census_region"], NAT_PCT).toPandas()], ignore_index=True)
SPARK_TABLES["rq2_region_profiles"] = (reg, ["census_region"])
SPARK_TABLES["rq2_region_dti_distribution"] = (share_table(dreg, ["census_region"], "dti_cat").toPandas(), ["census_region", "dti_cat"])
SPARK_TABLES["rq2_region_loan_type_mix"] = (share_table(dreg, ["census_region"], "loan_type_name").toPandas(),
                                            ["census_region", "loan_type_name"])
OUT_COLS = ["n", "income_kusd_median", "loan_usd_median", "lti_median", "share_dti_50plus_pct",
            "share_dti_missing_pct", "share_conventional_pct", "share_fha_pct"]
ro = pd.concat([indicator_profile(dreg, ["census_region", "decision_group"], NAT_PCT).toPandas(),
                indicator_profile(d_all, ["census_region", "decision_group"], NAT_PCT).toPandas()], ignore_index=True)
SPARK_TABLES["rq2_region_by_outcome"] = (ro[["census_region", "decision_group"] + OUT_COLS], ["census_region", "decision_group"])
stp = rate_ci(indicator_profile(d50, ["state", "census_region"], STATE_REF_PCT)).toPandas()
stp = stp[[c for c in stp.columns if c not in ("wilson_95_low_pct", "wilson_95_high_pct", "reportable")] +
          ["wilson_95_low_pct", "wilson_95_high_pct", "reportable"]]
stp["denial_rate_pct"] = 100 * stp["denied"] / stp["n"]          # same formula as indicator_profile
SPARK_TABLES["rq2_state_profiles_gap"] = (stp, ["state", "census_region"])
SPARK_TABLES["rq2_state_gap_correlations"] = (corr_with_gap(stp), ["indicator"])
top5, bottom5 = extreme_states(stp)
ext = []
for lab, sts in [("Top 5 gap", top5), ("Bottom 5 gap", bottom5), (ALL_STATES, None)]:
    part = d50 if sts is None else d50.filter(F.col("state").isin(sts))
    row = indicator_profile(part.withColumn("group", F.lit(lab)), ["group"], STATE_REF_PCT).toPandas()
    row.insert(1, "states", "all" if sts is None else ", ".join(sts))
    ext.append(row)
SPARK_TABLES["rq2_gap_extremes"] = (pd.concat(ext, ignore_index=True), ["group"])

# Gap expected from mix: stratum rates of the 50 states + DC joined back to each application, averaged by state
prev = (d50.groupBy("state").agg(F.count(F.lit(1)).alias("n"), F.sum("denied").cast("long").alias("denied"))
           .withColumn("denial_rate_pct", 100 * F.col("denied") / F.col("n"))
           .withColumn("raw_gap_pp", F.col("denial_rate_pct") - F.lit(STATE_REF_PCT)))
for tag, keys in [("loan_type", ["loan_type_name"]), ("loan_type_dti", ["loan_type_name", "dti_cat"])]:
    stratum = d50.groupBy(*keys).agg(F.avg(F.col("denied").cast("double")).alias("_r"))
    exp = (d50.select("state", *keys).join(stratum, keys).groupBy("state")
              .agg((100 * F.avg("_r") - F.lit(STATE_REF_PCT)).alias(f"exp_gap_{tag}_pp")))
    prev = (prev.join(exp, "state")
                .withColumn(f"resid_gap_{tag}_pp", F.col("raw_gap_pp") - F.col(f"exp_gap_{tag}_pp")))
prev_pd = prev.select("state", "n", "denied", "denial_rate_pct", "raw_gap_pp", "exp_gap_loan_type_pp", "resid_gap_loan_type_pp",
                      "exp_gap_loan_type_dti_pp", "resid_gap_loan_type_dti_pp").toPandas()
SPARK_TABLES["rq2_state_gap_mix_preview"] = (prev_pd, ["state"])
SPARK_TABLES["rq2_gap_mix_summary"] = (mix_summary(prev_pd), ["mix"])

print(f"{len(SPARK_TABLES)} tables computed in Spark in {time.perf_counter() - t_start:.0f}s")
for name, (t, keys) in SPARK_TABLES.items():
    print(f"  {name:38s} {len(t):>4} rows")

37 tables computed in Spark in 142s
  cohort_outcome_counts                     3 rows
  data_quality_missingness                  4 rows
  data_quality_dti_overall                  9 rows
  data_quality_numeric_quantiles            3 rows
  data_quality_original_flags               6 rows
  rq2_missingness_vs_denial                 4 rows
  rq2_cltv_missing_by_outcome               3 rows
  rq2_complete_case_vs_missing_category     6 rows
  rq2_financial_profiles                    8 rows
  rq2_missingness_by_outcome                2 rows
  rq2_dti_by_loan_type                     36 rows
  rq2_income_by_loan_type                  28 rows
  rq2_cltv_by_loan_type                    24 rows
  rq2_loan_amount_by_loan_type             24 rows
  rq2_dti_interpretation_focus             20 rows
  rq2_dti_x_cltv                           54 rows
  rq2_state_financial_profiles             51 rows
  rq2_msa_financial_profiles              134 rows
  rq1_loan_type_rates                       4 

In [16]:
# Quick look: region profiles (gap vs all records) and the mix summary (state gaps vs 50 states + DC)
show_cols = ["census_region", "n", "denial_rate_pct", "gap_pp", "income_kusd_median", "share_dti_50plus_pct", "share_fha_pct"]
print(SPARK_TABLES["rq2_region_profiles"][0][show_cols].round(2).to_string(index=False))
print()
print(SPARK_TABLES["rq2_gap_mix_summary"][0].round(3).to_string(index=False))

census_region       n  denial_rate_pct  gap_pp  income_kusd_median  share_dti_50plus_pct  share_fha_pct
      Midwest  666755             7.52   -1.29                92.0                  8.52          16.87
        South 1397598             9.61    0.80               107.0                 16.44          26.38
    Territory   12890            19.36   10.56                51.0                 10.99          51.12
         West  654252             7.73   -1.07               138.0                 14.76          20.30
    Northeast  401336             8.45   -0.36               125.0                  9.46          14.77
  All records 3140857             8.81    0.00               112.0                 13.53          21.69

                 mix  n_states  r_squared  spearman_rho  mean_abs_raw_gap_pp  mean_abs_remaining_gap_pp
           loan type        51      0.138         0.280                1.343                      1.263
loan type x DTI band        51      0.508         0.588        

In [17]:
# Save the Spark tables (local disk first, then copy to Drive)
for name, (t, keys) in SPARK_TABLES.items():
    t.to_csv(f"{OUT_LOCAL}/{name}.csv", index=False)
os.makedirs(OUT_DRIVE, exist_ok=True)
for fpath in glob.glob(f"{OUT_LOCAL}/*.csv"):
    _ = shutil.copy(fpath, OUT_DRIVE)
print("Saved", len(SPARK_TABLES), "tables to", OUT_DRIVE)

Saved 37 tables to /content/drive/MyDrive/hmda/results/rq2_spark


## Step 10. Correctness: Pandas vs PySpark
Reads the outputs of notebook 04 (`results/rq1_rq2_pandas/`; older runs: `GROUP7_RQ1_RQ2_FINAL_OUTPUTS*.zip`) and compares every table on its key columns:
- the **set of groups** must be identical (no group only in one tool);
- **counts** must be exactly equal; every other number must agree within a **relative tolerance of 1e-9** (1e-9 absolute near zero);
- text/boolean columns (`reportable`) must be identical.

In [18]:
def find_pandas_outputs():
    nb04 = f"{RESULTS_DIR}/rq1_rq2_pandas"                     # output folder of notebook 04
    if os.path.exists(f"{nb04}/rq2_region_profiles.csv"):
        return nb04
    if os.path.exists(f"{PANDAS_LOCAL}/rq2_financial_profiles.csv"):
        return PANDAS_LOCAL
    folders = [p for p in glob.glob(f"{DRIVE_DIR}/GROUP7_RQ1_RQ2_FINAL_OUTPUTS*") if os.path.isdir(p)]
    for p in folders:
        hits = glob.glob(f"{p}/**/rq2_financial_profiles.csv", recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    zips = sorted(glob.glob(f"{DRIVE_DIR}/GROUP7_RQ1_RQ2_FINAL_OUTPUTS*.zip"), key=os.path.getmtime)
    assert zips, "Pandas outputs not found - put GROUP7_RQ1_RQ2_FINAL_OUTPUTS.zip in the hmda folder"
    with zipfile.ZipFile(zips[-1]) as z:
        z.extractall(PANDAS_LOCAL)
    hits = glob.glob(f"{PANDAS_LOCAL}/**/rq2_financial_profiles.csv", recursive=True)
    return os.path.dirname(hits[0])

PANDAS_DIR = find_pandas_outputs()
print("Pandas outputs:", PANDAS_DIR)

def compare_table(name, s, keys, rtol=1e-9, atol=1e-9):
    p = pd.read_csv(f"{PANDAS_DIR}/{name}.csv", dtype={k: str for k in keys})
    s = s.copy()
    for k in keys:
        s[k] = s[k].astype(str)
    col_diff = sorted(set(p.columns) ^ set(s.columns))
    if col_diff:
        return "MISMATCH", f"columns only in one tool: {col_diff}"
    mrg = p.merge(s, on=keys, how="outer", suffixes=("_pd", "_sp"), indicator=True)
    only_pd = mrg.loc[mrg["_merge"] == "left_only", keys].values.tolist()
    only_sp = mrg.loc[mrg["_merge"] == "right_only", keys].values.tolist()
    if only_pd or only_sp:
        return "MISMATCH", f"groups only in Pandas: {only_pd[:5]} | only in Spark: {only_sp[:5]}"
    problems = []
    for c in [c for c in p.columns if c not in keys]:
        a, b = mrg[f"{c}_pd"], mrg[f"{c}_sp"]
        if pd.api.types.is_bool_dtype(a) or not pd.api.types.is_numeric_dtype(a):
            bad = (a.astype(str) != b.astype(str))
        else:
            a, b = a.astype(float).to_numpy(), b.astype(float).to_numpy()
            bad = ~np.isclose(a, b, rtol=rtol, atol=atol, equal_nan=True)
        if bad.any():
            i = int(np.flatnonzero(np.asarray(bad))[0])
            problems.append(f"{c}: {int(np.sum(bad))} cells differ, e.g. {mrg.loc[i, keys].tolist()} "
                            f"pandas={mrg.loc[i, f'{c}_pd']} spark={mrg.loc[i, f'{c}_sp']}")
    if problems:
        return "MISMATCH", " ; ".join(problems)
    return "OK", f"{len(p)} groups, {len(p.columns) - len(keys)} value columns identical"

results = []
for name, (t, keys) in SPARK_TABLES.items():
    status, details = compare_table(name, t, keys)
    results.append({"test": f"Pandas-PySpark: {name}", "status": status, "details": details})
    print(f"{status:9s} {name:34s} {details}")

corr = pd.DataFrame(results)
n_bad = int((corr["status"] != "OK").sum())
corr.loc[len(corr)] = {"test": "Pandas-PySpark: all tables",
                       "status": "OK" if n_bad == 0 else "MISMATCH",
                       "details": f"{len(results) - n_bad} of {len(results)} tables identical"}
corr.to_csv(f"{OUT_LOCAL}/pandas_spark_correctness_rq2.csv", index=False)
_ = shutil.copy(f"{OUT_LOCAL}/pandas_spark_correctness_rq2.csv", OUT_DRIVE)
nb04_dir = f"{RESULTS_DIR}/rq1_rq2_pandas"
if os.path.isdir(nb04_dir):                                    # notebook 04 lists this file in its outputs
    corr.to_csv(f"{nb04_dir}/pandas_spark_correctness.csv", index=False)
print("\n=> PANDAS = SPARK for all tables" if n_bad == 0 else f"\n=> {n_bad} tables MISMATCH - see details above")

Pandas outputs: /content/drive/MyDrive/hmda/results/rq1_rq2_pandas
OK        cohort_outcome_counts              3 groups, 2 value columns identical
OK        data_quality_missingness           4 groups, 3 value columns identical
OK        data_quality_dti_overall           9 groups, 3 value columns identical
OK        data_quality_numeric_quantiles     3 groups, 8 value columns identical
OK        data_quality_original_flags        6 groups, 1 value columns identical
OK        rq2_missingness_vs_denial          4 groups, 5 value columns identical
OK        rq2_cltv_missing_by_outcome        3 groups, 3 value columns identical
OK        rq2_complete_case_vs_missing_category 6 groups, 8 value columns identical
OK        rq2_financial_profiles             8 groups, 17 value columns identical
OK        rq2_missingness_by_outcome         2 groups, 7 value columns identical
OK        rq2_dti_by_loan_type               36 groups, 6 value columns identical
OK        rq2_income_by_loan_type    

## Step 11. Summary of all checks

In [19]:
chk = pd.DataFrame(CHECKS, dtype=object)
print(chk.to_string(index=False))
n_mis = int((chk["status"] != "OK").sum())
print(f"\nData checks: {len(chk) - n_mis} of {len(chk)} OK")
print(f"Pandas-PySpark: {len(results) - n_bad} of {len(results)} tables identical")
chk.to_csv(f"{OUT_LOCAL}/profiling_checks_spark.csv", index=False)
_ = shutil.copy(f"{OUT_LOCAL}/profiling_checks_spark.csv", OUT_DRIVE)

                                             check      got expected status
                            raw: number of columns       99       99     OK
                            raw: number of records 12259199 12259199     OK
                  scope: after filter action_taken  8661779  8661779     OK
                scope: after filter occupancy_type  7792568  7792568     OK
                  scope: after filter loan_purpose  3811298  3811298     OK
                   scope: after filter lien_status  3641962  3641962     OK
     scope: after filter derived_dwelling_category  3237835  3237835     OK
              scope: after filter reverse_mortgage  3168105  3168105     OK
       scope: after filter open_end_line_of_credit  3145390  3145390     OK
scope: after filter business_or_commercial_purpose  3140857  3140857     OK
                             scope: originated (1)  2775269  2775269     OK
                 scope: approved, not accepted (2)    88984    88984     OK
            

## How to read the result
- **Steps 3-6** must show only `OK`: same raw file, same scope and the same cleaning counts as `cleaning_log_spark.json`.
- **Step 10** is the RQ2 part of the correctness comparison (it replaces the `NOT RUN` line in the Pandas `pandas_spark_correctness.csv`). If a table shows `MISMATCH`, the details name the column and the first group that differs; the usual causes are a band edge (`<` vs `<=`) or a different minimum group size (MSA cut-off must be 5,000 in both notebooks).
- Outputs on Drive: `results/rq2_spark/` (14 Spark tables, `pandas_spark_correctness_rq2.csv`, `profiling_checks_spark.csv`).